# Inspección Automatizada de Texturas en Llantas Industriales
Este cuaderno unifica el flujo completo de experimentación, desarrollo de arquitecturas, estudio de ablación e interpretabilidad visual para la detección de fallas estructurales (`cracked` vs `normal`).

### Inicialización del Entorno y Verificación de Hardware

In [ ]:
import torch
import os
import numpy as np

# Configuración de consistencia y hardware
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Motor de ejecución configurado en: {device}")
if torch.cuda.is_available():
    print(f"🔥 GPU Detectada: {torch.cuda.get_device_name(0)}")

## 2. Configuración de DataLoaders Estructurados y Mitigación de Desbalance
Conexión directa con la topología nativa del dataset extraído y aplicación de un `WeightedRandomSampler` para equilibrar la influencia de los gradientes durante el entrenamiento.

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler

ruta_base = "/content/dataset_bruto/Tire Textures"
train_dir = os.path.join(ruta_base, 'training_data')
val_dir = os.path.join(ruta_base, 'testing_data')

# Transformaciones con Data Augmentation morfológico para texturas
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
val_dataset = datasets.ImageFolder(root=val_dir, transform=val_transforms)

# Cálculo de pesos analíticos para mitigar el desbalance
class_counts = np.bincount(train_dataset.targets)
class_weights = 1.0 / class_counts
sample_weights = [class_weights[label] for label in train_dataset.targets]

sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ Clases detectadas del sistema: {train_dataset.classes}")
print(f"✅ Inventario balanceado configurado para producción.")

## 3. Auditoría Visual y Control de Dimensiones del Lote
Extracción de una muestra de control estocástica para verificar la correcta ejecución de las transformaciones espaciales.

In [ ]:
import matplotlib.pyplot as plt
import torchvision

def desnormalizar(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return torch.clamp(tensor * std + mean, 0, 1)

imagenes_batch, etiquetas_batch = next(iter(train_loader))

fig = plt.figure(figsize=(16, 4))
grid = torchvision.utils.make_grid(imagenes_batch[:8], nrow=4)
plt.imshow(desnormalizar(grid).permute(1, 2, 0))
plt.title("Muestra de Control del DataLoader (Primeros 8 Elementos del Batch)")
plt.axis('off')
plt.show()
print(f"📐 Dimensiones estructurales del tensor batch: {imagenes_batch.shape}")

## 4. Motor de Entrenamiento Base (Pipeline Estándar)
Algoritmo cíclico de optimización para el cálculo de pérdidas y retropropagación de gradientes sin modificadores térmicos.

In [ ]:
import time
import copy

def entrenar_modelo(modelo, dataloaders, criterio, optimizador, num_epocas=10, nombre_guardado="modelo_base.pth"):
    inicio = time.time()
    mejores_pesos = copy.deepcopy(modelo.state_dict())
    mejor_precision = 0.0
    historial = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoca in range(num_epocas):
        print(f'\nÉpoca {epoca+1}/{num_epocas}')
        print('-' * 10)

        for fase in ['train', 'val']:
            if fase == 'train':
                modelo.train()
                loader = dataloaders[0]
            else:
                modelo.eval()
                loader = dataloaders[1]

            perdida_acumulada = 0.0
            aciertos_acumulados = 0

            for entradas, etiquetas in loader:
                entradas = entradas.to(device)
                etiquetas = etiquetas.to(device)
                optimizador.zero_grad()

                with torch.set_grad_enabled(fase == 'train'):
                    salidas = modelo(entradas)
                    _, preds = torch.max(salidas, 1)
                    perdida = criterio(salidas, etiquetas)
                    if fase == 'train':
                        perdida.backward()
                        optimizador.step()

                perdida_acumulada += perdida.item() * entradas.size(0)
                aciertos_acumulados += torch.sum(preds == etiquetas.data)

            perdida_ep = perdida_acumulada / len(loader.dataset)
            prec_ep = aciertos_acumulados.double() / len(loader.dataset)
            
            historial[f'{fase}_loss'].append(perdida_ep)
            historial[f'{fase}_acc'].append(prec_ep.item())
            print(f'{fase.capitalize()} - Pérdida: {perdida_ep:.4f} Precisión: {prec_ep:.4f}')

            if fase == 'val' and prec_ep > mejor_precision:
                mejor_precision = prec_ep
                mejores_pesos = copy.deepcopy(modelo.state_dict())
                torch.save(modelo.state_dict(), nombre_guardado)
                print(f"🌟 ¡Nuevo mejor modelo guardado con precisión de {mejor_precision:.4f}!")

    print(f'\nEntrenamiento completado en {(time.time() - inicio)//60:.0f}m {(time.time() - inicio)%60:.0f}s')
    modelo.load_state_dict(mejores_pesos)
    return modelo, historial

## 5. Arquitectura Convolucional Custom (Baseline desde Cero)
Diseño e instanciación de la red convolucional personalizada para el establecimiento de la línea base operativa.

In [ ]:
import torch.nn as nn
import torch.optim as optim

class CustomTextureCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(CustomTextureCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(torch.flatten(self.global_pool(self.features(x)), 1))

print("Instanciando CustomTextureCNN Baseline...")
modelo_base = CustomTextureCNN(num_classes=2).to(device)
criterio_base = nn.CrossEntropyLoss()
opt_base = optim.Adam(modelo_base.parameters(), lr=0.001)

# Descomentar para ejecutar el entrenamiento base
# modelo_base, hist_base = entrenar_modelo(modelo_base, [train_loader, val_loader], criterio_base, opt_base)

## 6. Estudio de Ablación Avanzado: Focal Loss y Dinámica de Aprendizaje
Implementación analítica de la función de pérdida adaptativa para optimizar la convergencia sobre ejemplos de texturas ambiguas.

In [ ]:
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

modelo_ablacion = CustomTextureCNN(num_classes=2).to(device)
criterio_focal = FocalLoss(gamma=2.0)
opt_ablacion = optim.Adam(modelo_ablacion.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt_ablacion, mode='min', factor=0.5, patience=2)

print("🔬 Arquitectura lista para evaluación del impacto del gradiente focal.")

### Bucle de Entrenamiento Ajustado con Monitoreo de Tasa de Aprendizaje (LR)

In [ ]:
def entrenar_con_scheduler(modelo, dataloaders, criterio, optimizador, scheduler, num_epocas=10, nombre_guardado="modelo_ablacion.pth"):
    inicio = time.time()
    mejores_pesos = copy.deepcopy(modelo.state_dict())
    mejor_precision = 0.0
    historial = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoca in range(num_epocas):
        print(f'\nÉpoca {epoca+1}/{num_epocas} (Estudio de Ablación)')
        print('-' * 10)

        for fase in ['train', 'val']:
            if fase == 'train':
                modelo.train()
                loader = dataloaders[0]
            else:
                modelo.eval()
                loader = dataloaders[1]

            perdida_acumulada = 0.0
            aciertos_acumulados = 0

            for entradas, etiquetas in loader:
                entradas = entradas.to(device)
                etiquetas = etiquetas.to(device)
                optimizador.zero_grad()

                with torch.set_grad_enabled(fase == 'train'):
                    salidas = modelo(entradas)
                    _, preds = torch.max(salidas, 1)
                    perdida = criterio(salidas, etiquetas)
                    if fase == 'train':
                        perdida.backward()
                        optimizador.step()

                perdida_acumulada += perdida.item() * entradas.size(0)
                aciertos_acumulados += torch.sum(preds == etiquetas.data)

            perdida_ep = perdida_acumulada / len(loader.dataset)
            prec_ep = aciertos_acumulados.double() / len(loader.dataset)
            
            if fase == 'train':
                historial['train_loss'].append(perdida_ep)
                historial['train_acc'].append(prec_ep.item())
            else:
                historial['val_loss'].append(perdida_ep)
                historial['val_acc'].append(prec_ep.item())
                scheduler.step(perdida_ep)
                
            print(f'{fase.capitalize()} - Pérdida: {perdida_ep:.4f} Precisión: {prec_ep:.4f}')
            if fase == 'val':
                print(f"Tasa de aprendizaje (LR actual): {optimizador.param_groups[0]['lr']}")
                if prec_ep > mejor_precision:
                    mejor_precision = prec_ep
                    mejores_pesos = copy.deepcopy(modelo.state_dict())
                    torch.save(modelo.state_dict(), nombre_guardado)
                    print(f"🌟 ¡Mejor modelo de ablación guardado (Precisión: {mejor_precision:.4f})!")

    print(f'\nEntrenamiento completado en {(time.time() - inicio)//60:.0f}m')
    modelo.load_state_dict(mejores_pesos)
    return modelo, historial

# Descomentar para ejecutar el estudio de ablación
# modelo_ablacion, hist_ab = entrenar_con_scheduler(modelo_ablacion, [train_loader, val_loader], criterio_focal, opt_ablacion, scheduler)

## 8. Transfer Learning de Alta Capacidad: ResNet-50
Carga de pesos preentrenados en ImageNet y congelamiento dinámico del extractor de características convolucionales profundas.

In [ ]:
import torchvision.models as models

class TransferLearningResNet(nn.Module):
    def __init__(self, num_classes=2):
        super(TransferLearningResNet, self).__init__()
        self.model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        
        # Bloqueo total de capas profundas preexistentes
        for param in self.model.parameters():
            param.requires_grad = False
            
        # Rediseño adaptativo del clasificador terminal
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_ftrs, num_classes)
        )

    def forward(self, x):
        return self.model(x)

modelo_resnet = TransferLearningResNet(num_classes=2).to(device)
criterio_resnet = nn.CrossEntropyLoss()
opt_resnet = optim.Adam(modelo_resnet.model.fc.parameters(), lr=0.001)

# Descomentar para lanzar el motor sobre la arquitectura profunda ResNet-50
# modelo_resnet, hist_resnet = entrenar_modelo(modelo_resnet, [train_loader, val_loader], criterio_resnet, opt_resnet, num_epocas=10, nombre_guardado="resnet50_mejor_modelo.pth")

## 9. Evaluación Multimétrica Rigurosa y Análisis Estadístico
Cálculo de matrices de confusión e inferencia matemática de curvas ROC y coeficientes AUC para el aseguramiento de la robustez del sistema.

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

def evaluar_modelo_riguroso(modelo, dataloader, nombre_modelo="ResNet-50"):
    modelo.eval()
    reales, preds_clase, probs_pos = [], [], []
    
    with torch.no_grad():
        for entradas, etiquetas in dataloader:
            entradas = entradas.to(device)
            salidas = modelo(entradas)
            probs = F.softmax(salidas, dim=1)
            
            reales.extend(etiquetas.numpy())
            preds_clase.extend(torch.argmax(salidas, 1).cpu().numpy())
            probs_pos.extend(probs[:, 1].cpu().numpy())
            
    reales, preds_clase, probs_pos = np.array(reales), np.array(preds_clase), np.array(probs_pos)
    nombres_clases = dataloader.dataset.classes
    
    print(classification_report(reales, preds_clase, target_names=nombres_clases))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    cm = confusion_matrix(reales, preds_clase)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1, xticklabels=nombres_clases, yticklabels=nombres_clases)
    ax1.set_title(f'Matriz de Confusión - {nombre_modelo}')
    
    fpr, tpr, _ = roc_curve(reales, probs_pos)
    ax2.plot(fpr, tpr, color='darkorange', lw=2, label=f'Curva ROC (AUC = {auc(fpr, tpr):.3f})')
    ax2.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    ax2.set_title(f'Curva ROC - {nombre_modelo}')
    ax2.legend(loc="lower right")
    plt.show()

## 10. Algoritmo de Interpretabilidad Visual Grad-CAM (Aporte Propio)
Interceptación matemática del flujo tensorial en capas congeladas utilizando hooks dinámicos para auditoría visual.

In [ ]:
import cv2
from PIL import Image

class GradCAM:
    def __init__(self, modelo, target_layer):
        self.modelo = modelo
        self.target_layer = target_layer
        self.activations = None
        self.target_layer.register_forward_hook(self.guardar_activacion)

    def guardar_activacion(self, module, input, output):
        self.activations = output
        if not self.activations.requires_grad:
            self.activations.requires_grad_(True)
        self.activations.retain_grad()

    def __call__(self, x, class_idx=None):
        self.modelo.eval()
        out = self.modelo(x)
        if class_idx is None:
            class_idx = torch.argmax(out, dim=1).item()
            
        self.modelo.zero_grad()
        out[0, class_idx].backward(retain_graph=True)
        
        gradients = self.activations.grad.cpu().data.numpy()[0]
        activations = self.activations.cpu().data.numpy()[0]
        
        weights = np.mean(gradients, axis=(1, 2))
        cam = np.zeros(activations.shape[1:], dtype=np.float32)
        for i, w in enumerate(weights):
            cam += w * activations[i, :, :]
            
        cam = np.maximum(cam, 0)
        cam = cv2.resize(cam, (x.shape[3], x.shape[2]))
        return (cam - np.min(cam)) / np.max(cam)

def analizar_llanta_gradcam(ruta_imagen, modelo, dataloader):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    img_pil = Image.open(ruta_imagen).convert('RGB')
    img_tensor = transform(img_pil).unsqueeze(0).to(device)
    
    target_layer = modelo.model.layer4[-1].conv3
    grad_cam = GradCAM(modelo, target_layer)
    cam = grad_cam(img_tensor)
    
    with torch.no_grad():
        salida = modelo(img_tensor)
        pred = torch.argmax(salida, 1).item()
        confianza = F.softmax(salida, dim=1)[0][pred].item() * 100
        
    img_np = np.array(img_pil.resize((224, 224))) / 255.0
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = np.float32(heatmap)[:, :, ::-1] / 255
    
    overlay = np.clip(heatmap * 0.5 + img_np * 0.5, 0, 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
    ax1.imshow(img_np); ax1.axis('off'); ax1.set_title("Textura Original")
    ax2.imshow(overlay); ax2.axis('off'); ax2.set_title(f"Grad-CAM: {dataloader.dataset.classes[pred]} ({confianza:.1f}%)")
    plt.show()

## 11. Auditoría Automatizada de Desempeño (Análisis de Errores)
Aislamiento algorítmico de los fallos predictivos estructurales para la identificación de sesgos por características dominantes (relieves y marcas).

In [ ]:
import random

def ejecutar_extraccion_fallos(modelo, dataloader):
    modelo.eval()
    errores = 0
    muestras = list(dataloader.dataset.samples)
    random.seed(42)
    random.shuffle(muestras)
    
    transformacion_prueba = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    for ruta_img, etiqueta_real in muestras:
        if errores >= 5: break
        img_pil = Image.open(ruta_img).convert('RGB')
        tensor = transformacion_prueba(img_pil).unsqueeze(0).to(device)
        
        with torch.no_grad():
            pred = torch.argmax(modelo(tensor), 1).item()
            
        if pred != etiqueta_real:
            print(f"\n❌ FALLO DETECTADO [{errores+1}/5] | Archivo: {ruta_img.split('/')[-1]}")
            print(f"Realidad: {dataloader.dataset.classes[etiqueta_real]} | Predicción: {dataloader.dataset.classes[pred]}")
            analizar_llanta_gradcam(ruta_img, modelo, dataloader)
            errores += 1
            
print("🚀 Notebook cargado con la estructura científica unificada completa.")